# 08. 일기 감정 분석 — 문맥 검증형 직접 감정 탐지

**작성자: 김민서**  
**작성일: 2026-11-23**

## 07 대비 업데이트 및 수정 사항

1. **`혼합` 대표 감정 제거**
   - 07의 `MIXED_MARGIN` 판정을 제거했습니다.
   - 하루 대표 감정은 **행복 / 기대 / 슬픔 / 분노 / 불안 / 피로 / 중립** 중 하나로 유지합니다.
   - 여러 감정은 문장별 결과와 세부 감정 Top 5에서 함께 보여줍니다.

2. **직접 언급 감정의 앞뒤 문맥 검사 추가**
   - 감정 단어가 보인다는 이유만으로 바로 현재 감정으로 인정하지 않습니다.
   - **부정 표현**: `전혀 행복하지 않았다`, `안 피곤하다`, `걱정하지 않았다`
   - **가정·희망·예상**: `합격하면 행복할 것 같다`, `행복했으면 좋겠다`, `뿌듯할 줄 알았다`
   - **타인의 감정**: `친구가 행복해 보였다`
   - **인용·전달 표현**: `친구가 "나는 행복하다"고 말했다`
   - **명시적인 과거 회상**: `예전에는 행복했다`
   - 위 문맥에 해당하면 직접 감정 후보에서 제외합니다.

3. **문맥 검증을 통과한 명시적 감정은 세부 감정에 포함**
   - KoBERT 확률이 감정별 임계값보다 낮더라도, 현재 사용자의 실제 감정 표현으로 검증되면 최종 다중 감정 목록에 포함합니다.
   - 출력에서는 `*`로 표시합니다.

4. **대표 감정과 명시적 감정의 역할 분리**
   - 대표 감정은 기존처럼 모델의 그룹 강도와 긍정·부정 점수로 결정합니다.
   - 직접 언급 감정은 세부 감정의 누락을 보완하지만 대표 감정을 강제로 덮어쓰지 않습니다.

5. **07 방식과 08 방식 비교 검증 셀 추가**
   - 같은 문장을 07의 단순 키워드 방식과 08의 문맥 검증 방식으로 비교합니다.
   - 예상 결과를 자동 검증하는 assertion을 포함했습니다.

6. **유지한 부분**
   - 모델은 `kobert_kote_nall_e5.pt`를 그대로 사용하며 **재학습하지 않습니다**.
   - 44개 KOTE 감정, 감정별 임계값 보정, 예측 캐시, 문장 분리, 긍정·부정 그래프, 감정 캘린더 구조를 유지합니다.



In [ ]:
# 1. 기본 설정
import re
import calendar
from pathlib import Path
from datetime import date
from IPython.display import display

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle, Patch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
model_dir = project_root / "models"
data_dir = project_root / "data"
output_dir = project_root / "outputs"
output_dir.mkdir(exist_ok=True)

for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(font_name == item.name for item in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = font_name
        break

plt.rcParams["axes.unicode_minus"] = False

print("project_root:", project_root)
print("font:", plt.rcParams["font.family"])


In [ ]:
# 2. 설정
MODEL_FILE = "kobert_kote_nall_e5.pt"
DIARY_FILE = data_dir / "sample_diary.csv"

THRESHOLD = None
SPLIT_CONNECTIVES = True
GROUP_TOP_K = 2
NEUTRAL_MIN_SCORE = 0.35

# 긍정/부정 그룹 수가 달라도 같은 기준으로 비교
POLARITY_AGG = "max"  # "mean" 또는 "max"

# 07에서 유지한 감정별 임계값 보정
USE_CLASS_THRESHOLD = True
CLASS_TH_GRID = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
CLASS_TH_MIN_POS = 30
RANK_BY_BASELINE = False

# 08 문맥 검증형 직접 감정 처리
USE_CONTEXT_MENTION = True
PRIORITIZE_CONTEXT_MENTION = True
INCLUDE_VALID_MENTION_BELOW_THRESHOLD = True
CONTEXT_WINDOW = 48

# 08에서는 혼합 대표 감정을 사용하지 않음
DISPLAY_EMOTIONS = ["행복", "기대", "슬픔", "분노", "불안", "피로", "중립"]


In [ ]:
# 3. KOTE 44개 세부 감정 + 7개 대표 감정 그룹
LABELS = [
    '불평/불만','환영/호의','감동/감탄','지긋지긋','고마움','슬픔','화남/분노','존경',
    '기대감','우쭐댐/무시함','안타까움/실망','비장함','의심/불신','뿌듯함','편안/쾌적',
    '신기함/관심','아껴주는','부끄러움','공포/무서움','절망','한심함','역겨움/징그러움',
    '짜증','어이없음','없음','패배/자기혐오','귀찮음','힘듦/지침','즐거움/신남','깨달음',
    '죄책감','증오/혐오','흐뭇함(귀여움/예쁨)','당황/난처','경악','부담/안 내킴','서러움',
    '재미없음','불쌍함/연민','놀람','행복','불안/걱정','기쁨','안심/신뢰'
]

GROUPS = {
    "기쁨·만족": [
        '환영/호의','감동/감탄','고마움','존경','뿌듯함','편안/쾌적','아껴주는',
        '흐뭇함(귀여움/예쁨)','행복','기쁨','즐거움/신남','안심/신뢰'
    ],
    "기대·관심": ['기대감','신기함/관심','깨달음','비장함'],
    "슬픔·실망": [
        '슬픔','안타까움/실망','절망','서러움','불쌍함/연민',
        '패배/자기혐오','죄책감','부끄러움'
    ],
    "분노·불만": [
        '불평/불만','화남/분노','짜증','어이없음','한심함',
        '역겨움/징그러움','증오/혐오','의심/불신','우쭐댐/무시함'
    ],
    "불안·당황": ['공포/무서움','불안/걱정','당황/난처','경악','놀람','부담/안 내킴'],
    "피로·지루함": ['지긋지긋','귀찮음','힘듦/지침','재미없음'],
    "없음": ['없음']
}

DISPLAY_EMOTION = {
    "기쁨·만족": "행복",
    "기대·관심": "기대",
    "슬픔·실망": "슬픔",
    "분노·불만": "분노",
    "불안·당황": "불안",
    "피로·지루함": "피로",
    "없음": "중립"
}

POSITIVE_GROUPS = ["기쁨·만족", "기대·관심"]
NEGATIVE_GROUPS = ["슬픔·실망", "분노·불만", "불안·당황", "피로·지루함"]
GROUP_NAMES = list(GROUPS)
GROUP_IDX = {group: [LABELS.index(emotion) for emotion in emotions]
             for group, emotions in GROUPS.items()}
LABEL_GROUP = {idx: group for group, indices in GROUP_IDX.items() for idx in indices}

EMOTION_COLORS = {
    "행복": "#F2C94C",
    "기대": "#5DBB8F",
    "슬픔": "#5B8DEF",
    "분노": "#E5534B",
    "불안": "#9B6FD3",
    "피로": "#8D7B68",
    "중립": "#BDBDBD"
}

all_idx = sum(GROUP_IDX.values(), [])
assert sorted(all_idx) == list(range(44)), "44개 감정 매핑을 확인하세요."
assert set(DISPLAY_EMOTIONS) == set(EMOTION_COLORS), "대표 감정과 색상 매핑을 확인하세요."

print("Positive:", POSITIVE_GROUPS)
print("Negative:", NEGATIVE_GROUPS)
print("Display:", DISPLAY_EMOTIONS)


In [ ]:
# 4. 최종 KoBERT 모델 불러오기
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel

MODEL_NAME = "skt/kobert-base-v1"
tokenizer = KoBERTTokenizer.from_pretrained(MODEL_NAME)

assert tokenizer.encode("한국어 모델을 공유합니다.") == [
    2, 4958, 6855, 2046, 7088, 1050, 7843, 54, 3
], "KoBERT 토크나이저가 정상적으로 로드되지 않았습니다."


class KoBERTMultiLabelClassifier(nn.Module):
    def __init__(self, num_labels=44, dropout=0.2):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        cls_vector = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls_vector))


model_path = model_dir / MODEL_FILE
if not model_path.exists():
    raise FileNotFoundError(
        f"모델 파일이 없습니다: {model_path}\n"
        f"models 폴더에 {MODEL_FILE}을 넣어주세요."
    )

ckpt = torch.load(model_path, map_location=device, weights_only=False)
model = KoBERTMultiLabelClassifier().to(device)

if isinstance(ckpt, dict) and "state_dict" in ckpt:
    model.load_state_dict(ckpt["state_dict"])
    TH = THRESHOLD if THRESHOLD is not None else ckpt.get("threshold", 0.6)
    MAX_LENGTH = ckpt.get("max_length", 64)
else:
    model.load_state_dict(ckpt)
    TH = THRESHOLD if THRESHOLD is not None else 0.6
    MAX_LENGTH = 64

model.eval()
print("모델:", MODEL_FILE)
print("threshold:", TH)
print("max_length:", MAX_LENGTH)


In [ ]:
# 5. 여러 문장 감정 확률 예측 + 캐시
_pred_cache = {}


@torch.no_grad()
def _predict_raw(sentences, batch_size=32):
    if not sentences:
        return np.zeros((0, 44), dtype=np.float32)

    outputs = []
    for start in range(0, len(sentences), batch_size):
        batch_sentences = sentences[start:start + batch_size]
        encoded = tokenizer(
            batch_sentences,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        logits = model(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"],
            token_type_ids=encoded.get("token_type_ids")
        )
        outputs.append(torch.sigmoid(logits).cpu().numpy())

    return np.concatenate(outputs, axis=0)


def predict_sentences(sentences, batch_size=32):
    """같은 문장은 한 번만 예측한다."""
    todo = [sentence for sentence in dict.fromkeys(sentences) if sentence not in _pred_cache]
    if todo:
        for sentence, probability in zip(todo, _predict_raw(todo, batch_size)):
            _pred_cache[sentence] = probability

    if not sentences:
        return np.zeros((0, 44), dtype=np.float32)
    return np.stack([_pred_cache[sentence] for sentence in sentences])


In [ ]:
# 5-1. 검증셋으로 감정별 임계값과 기준 확률 계산
# 처음 한 번은 val/test 10,000문장을 예측하고 outputs/calibration_*.npz에 저장한다.
raw_dir = data_dir / "raw"
calib_path = output_dir / f"calibration_{Path(MODEL_FILE).stem}.npz"


def load_kote(name):
    path = raw_dir / f"{name}.tsv"
    if not path.exists():
        raise FileNotFoundError(f"KOTE 파일이 없습니다: {path}")

    frame = pd.read_csv(path, sep="\t", header=None, names=["id", "text", "labels"])
    labels = np.zeros((len(frame), 44), dtype=np.int64)
    for row_index, value in enumerate(frame["labels"].astype(str)):
        labels[row_index, [int(item) for item in value.split(",")]] = 1
    return frame["text"].astype(str).tolist(), labels


def f1_scores(y_true, y_pred):
    tp = (y_true * y_pred).sum(0)
    fp = ((1 - y_true) * y_pred).sum(0)
    fn = (y_true * (1 - y_pred)).sum(0)
    denominator = 2 * tp + fp + fn
    per_class = np.where(denominator > 0, 2 * tp / np.maximum(denominator, 1), 0.0)
    micro = 2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)
    return per_class, float(micro), float(per_class.mean())


val_texts, val_y = load_kote("val")
test_texts, test_y = load_kote("test")

if calib_path.exists():
    saved = np.load(calib_path)
    val_probs = saved["val_probs"]
    test_probs = saved["test_probs"]
    print("저장된 예측 불러옴:", calib_path.name)
else:
    print("val/test 예측 중... 최초 실행 시에만 수행합니다.")
    val_probs = _predict_raw(val_texts, batch_size=64)
    test_probs = _predict_raw(test_texts, batch_size=64)
    np.savez_compressed(calib_path, val_probs=val_probs, test_probs=test_probs)
    print("저장:", calib_path.name)

assert val_probs.shape == (len(val_texts), 44)
assert test_probs.shape == (len(test_texts), 44)

grid = sorted(set(CLASS_TH_GRID) | {float(TH)})
f1_grid = np.stack([
    f1_scores(val_y, (val_probs >= threshold).astype(np.int64))[0]
    for threshold in grid
])
CLASS_TH = np.array([grid[index] for index in f1_grid.argmax(0)], dtype=np.float32)
CLASS_TH[val_y.sum(0) < CLASS_TH_MIN_POS] = TH
GLOBAL_TH = np.full(44, TH, dtype=np.float32)
TH_VEC = CLASS_TH if USE_CLASS_THRESHOLD else GLOBAL_TH

BASE_MEAN = val_probs.mean(0)
BASE_STD = val_probs.std(0) + 1e-6


def rank_order(probabilities):
    score = ((probabilities - BASE_MEAN) / BASE_STD
             if RANK_BY_BASELINE else probabilities)
    return np.argsort(score)[::-1]


comparison_rows = []
for data_name, y_true, probabilities in [("val", val_y, val_probs), ("test", test_y, test_probs)]:
    for method_name, thresholds in [(f"공통 임계값 {TH}", GLOBAL_TH), ("감정별 임계값", CLASS_TH)]:
        prediction = (probabilities >= thresholds).astype(np.int64)
        _, micro, macro = f1_scores(y_true, prediction)
        comparison_rows.append({
            "데이터": data_name,
            "방식": method_name,
            "micro-F1": round(micro, 4),
            "macro-F1": round(macro, 4),
            "문장당 예측 감정 수": round(prediction.sum(1).mean(), 2)
        })

threshold_compare = pd.DataFrame(comparison_rows)
print(threshold_compare.to_string(index=False))
print("사용 중:", "감정별 임계값" if USE_CLASS_THRESHOLD else f"공통 임계값 {TH}")

f1_global = f1_scores(test_y, (test_probs >= GLOBAL_TH).astype(np.int64))[0]
f1_class = f1_scores(test_y, (test_probs >= CLASS_TH).astype(np.int64))[0]
class_table = pd.DataFrame({
    "감정": LABELS,
    "val 정답 수": val_y.sum(0),
    "평소 확률": BASE_MEAN.round(3),
    "임계값": CLASS_TH.round(2),
    "test F1(공통)": f1_global.round(3),
    "test F1(감정별)": f1_class.round(3)
})
class_table["변화"] = (class_table["test F1(감정별)"] - class_table["test F1(공통)"]).round(3)
class_table.to_csv(
    output_dir / f"class_thresholds_{Path(MODEL_FILE).stem}.csv",
    index=False,
    encoding="utf-8-sig"
)
threshold_compare.to_csv(
    output_dir / f"threshold_compare_{Path(MODEL_FILE).stem}.csv",
    index=False,
    encoding="utf-8-sig"
)
class_table.sort_values("변화", ascending=False).head(10)


In [ ]:
# 5-2. 08 문맥 검증형 직접 감정 탐지
EMOTION_WORDS = {
    '뿌듯함':        r'뿌듯',
    '불안/걱정':     r'걱정|불안|초조',
    '힘듦/지침':     r'피곤|지쳤|지친|지치|힘들|힘든',
    '재미없음':      r'지루|재미없|심심',
    '죄책감':        r'미안|죄책|죄송',
    '고마움':        r'고마|고맙|감사',
    '짜증':          r'짜증',
    '행복':          r'행복',
    '기쁨':          r'기쁘|기뻤|기쁜',
    '슬픔':          r'슬프|슬펐|슬픈|우울',
    '화남/분노':     r'화가 나|화났|화나서|화난|화나는|열받',
    '공포/무서움':   r'무섭|무서|두렵|두려',
    '귀찮음':        r'귀찮',
    '부끄러움':      r'부끄|창피|민망',
    '안타까움/실망': r'실망|아쉽|아쉬|안타깝|안타까',
    '기대감':        r'기대',
    '서러움':        r'서럽|서러',
    '당황/난처':     r'당황|난처',
    '놀람':          r'놀랐|놀라|깜짝',
    '즐거움/신남':   r'즐거|즐겁|신나|신났|신난|재밌|재미있',
    '편안/쾌적':     r'(?<!불)편안|(?<!불)편하|(?<!불)편했|쾌적',
    '안심/신뢰':     r'안심|다행',
    '지긋지긋':      r'지긋지긋|지겹|지겨',
    '어이없음':      r'어이없|황당',
    '부담/안 내킴':  r'부담',
    '감동/감탄':     r'감동',
    '절망':          r'절망|막막',
    '한심함':        r'한심',
    '의심/불신':     r'의심',
    '존경':          r'존경',
    '깨달음':        r'깨달'
}

assert set(EMOTION_WORDS) <= set(LABELS)
_EMOTION_RE = {LABELS.index(emotion): re.compile(pattern)
               for emotion, pattern in EMOTION_WORDS.items()}

# 07에서 사용한 단순 부정 검사: 비교용으로 유지
_LEGACY_NEG_BEFORE = re.compile(r'(?:^|\s)(안|못)\s?$')
_LEGACY_NEG_AFTER = re.compile(r'^\S{0,3}지\s?(않|못)')

# 08 문맥 검사 규칙
_NEG_PREFIX_RE = re.compile(
    r'(?:전혀|하나도|별로|그다지|도무지|결코|절대|안|못)\s*(?:\S+\s*){0,2}$'
)
_NEG_SUFFIX_RE = re.compile(r'^\s*\S{0,6}?지\s*(?:않|못|아니|없)')
_MODAL_SUFFIX_RE = re.compile(
    r'^\s*(?:'
    r'(?:하|했|였|이었|일|될|적일|적이었)?으면\s*(?:좋겠|한다|바라|좋을)'
    r'|(?:하|할|일|될|적일)\s*것\s*(?:같|이라|일)'
    r'|(?:하|할|일|될)\s*줄\s*(?:알|몰)'
    r'|(?:하|할|일|될)\s*(?:듯|지도|까)'
    r'|(?:하|하고)\s*싶'
    r'|(?:하|하길|기를|길)\s*(?:바라|원하)'
    r'|(?:하|할|일|될)\s*수(?:도)?\s*(?:있|없)'
    r')'
)
_CONDITIONAL_PREFIX_RE = re.compile(r'(?:면|라면|다면|하면)\s*(?:정말\s*)?$')
_REPORT_SUFFIX_RE = re.compile(
    r'^\s*\S{0,5}?(?:라고|다고|고)\s*(?:말|했|하였다|생각|들었|썼|적었)'
)
_META_SUFFIX_RE = re.compile(r'^\s*(?:이라는|란)\s*(?:단어|감정|표현)')
_DISTANT_PAST_RE = re.compile(r'(?:예전에는|과거에는|작년에는|그때는|한때는|옛날에는)')
_THIRD_SUBJECT_RE = re.compile(
    r'(?:친구|엄마|아빠|부모님|동생|형|누나|언니|오빠|선생님|교수님|그녀|그 사람|사람들|친구들|걔|상대|팀원|동기)'
    r'(?:이|가|은|는|도)'
)
_FIRST_SUBJECT_RE = re.compile(
    r'(?:나|내가|나는|나도|저|제가|저는|저도|우리|우리가|우리는|우리도)(?:\s|$)'
)


def legacy_07_mentioned_labels(sentence):
    """07의 직접 언급 방식: 감정 키워드 + 가까운 부정 표현만 제외."""
    found = set()
    for idx, pattern in _EMOTION_RE.items():
        for match in pattern.finditer(sentence):
            if (_LEGACY_NEG_BEFORE.search(sentence[:match.start()])
                    or _LEGACY_NEG_AFTER.match(sentence[match.end():])):
                continue
            found.add(idx)
            break
    return found


def _inside_quotes(text, position):
    for open_quote, close_quote in [("“", "”"), ("‘", "’")]:
        last_open = text.rfind(open_quote, 0, position)
        last_close = text.rfind(close_quote, 0, position)
        if last_open > last_close and text.find(close_quote, position) != -1:
            return True

    for quote in ['"', "'"]:
        if text[:position].count(quote) % 2 == 1:
            return True
    return False


def _current_clause_prefix(text, start):
    """감정 표현 앞에서 가장 가까운 문장/대조절 경계 이후만 반환한다."""
    prefix = text[:start]
    boundaries = []
    for token in [".", "!", "?", "~", "\n", "지만", "그러나", "그런데", "반면"]:
        position = prefix.rfind(token)
        if position >= 0:
            boundaries.append(position + len(token))
    cut = max(boundaries) if boundaries else 0
    return prefix[cut:]


def _subject_segment(clause_prefix):
    """원인절 뒤의 새로운 감정절을 분리해 가장 가까운 주체만 본다."""
    parts = re.split(
        r'\S{0,12}(?:해서|아서|어서|여서|고서|는데|므로|니까)\s*',
        clause_prefix
    )
    return parts[-1]


def _nearest_subject_type(text, start):
    segment = _subject_segment(_current_clause_prefix(text, start))
    third_positions = [match.start() for match in _THIRD_SUBJECT_RE.finditer(segment)]
    first_positions = [match.start() for match in _FIRST_SUBJECT_RE.finditer(segment)]

    if third_positions and (not first_positions or max(third_positions) > max(first_positions)):
        return "third"
    if first_positions:
        return "first"
    return None


def _validate_context(sentence, start, end):
    """감정 표현이 현재 작성자의 실제 감정인지 규칙 기반으로 검증한다."""
    clause_prefix = _current_clause_prefix(sentence, start)
    local_prefix = clause_prefix[-CONTEXT_WINDOW:]
    local_suffix = sentence[end:end + CONTEXT_WINDOW]

    if _NEG_PREFIX_RE.search(local_prefix) or _NEG_SUFFIX_RE.match(local_suffix):
        return False, "부정 표현"
    if _inside_quotes(sentence, start):
        return False, "인용 표현"
    if _nearest_subject_type(sentence, start) == "third":
        return False, "타인 감정"
    if _REPORT_SUFFIX_RE.match(local_suffix) or _META_SUFFIX_RE.match(local_suffix):
        return False, "인용·전달/메타 표현"
    if _DISTANT_PAST_RE.search(clause_prefix):
        return False, "과거 회상"
    if (_MODAL_SUFFIX_RE.match(local_suffix)
            or _CONDITIONAL_PREFIX_RE.search(local_prefix)):
        return False, "가정·희망·예상"

    return True, "현재 감정 표현"


def context_mention_details(sentence):
    """문장의 모든 명시적 감정 표현과 수용/제외 사유를 반환한다."""
    details = []
    for idx, pattern in _EMOTION_RE.items():
        for match in pattern.finditer(sentence):
            accepted, reason = _validate_context(sentence, match.start(), match.end())
            details.append({
                "idx": idx,
                "emotion": LABELS[idx],
                "matched": match.group(0),
                "start": match.start(),
                "end": match.end(),
                "accepted": accepted,
                "reason": reason
            })
    return sorted(details, key=lambda item: (item["start"], item["idx"]))


def context_mentioned_labels(sentence):
    if not USE_CONTEXT_MENTION:
        return set()
    return {item["idx"] for item in context_mention_details(sentence) if item["accepted"]}


def display_order(probabilities, sentence):
    base = list(rank_order(probabilities))
    if not PRIORITIZE_CONTEXT_MENTION:
        return base
    mentioned = context_mentioned_labels(sentence)
    return [idx for idx in base if idx in mentioned] + [idx for idx in base if idx not in mentioned]


In [ ]:
# 5-3. 07 방식과 08 문맥 검증 방식 비교
CONTEXT_VALIDATION_CASES = [
    ("현재 감정", "나는 지금 정말 행복하다.", {"행복"}),
    ("가정", "합격하면 행복할 것 같다.", set()),
    ("희망", "행복했으면 좋겠다.", set()),
    ("현재 감정", "시험을 잘 봐서 정말 뿌듯했다.", {"뿌듯함"}),
    ("예상/회고", "뿌듯할 줄 알았다.", set()),
    ("현재 감정", "모든 게 절망적이었다.", {"절망"}),
    ("예상", "절망할 것 같았다.", set()),
    ("부정", "전혀 행복하지 않았다.", set()),
    ("부정", "하나도 안 피곤하다.", set()),
    ("부정", "걱정하지 않았다.", set()),
    ("타인 감정", "친구가 행복해 보였다.", set()),
    ("인용", '친구가 "나는 행복하다"고 말했다.', set()),
    ("과거→현재", "예전에는 행복했지만 지금은 힘들다.", {"힘듦/지침"}),
    ("타인→현재", "친구가 행복해서 나도 기뻤다.", {"기쁨"}),
]

validation_rows = []
for category, sentence, expected in CONTEXT_VALIDATION_CASES:
    legacy = {LABELS[idx] for idx in legacy_07_mentioned_labels(sentence)}
    details = context_mention_details(sentence)
    accepted = {item["emotion"] for item in details if item["accepted"]}
    excluded = [f"{item['emotion']}({item['reason']})" for item in details if not item["accepted"]]

    validation_rows.append({
        "유형": category,
        "문장": sentence,
        "07 직접 언급": ", ".join(sorted(legacy)) or "없음",
        "08 인정 감정": ", ".join(sorted(accepted)) or "없음",
        "08 제외 및 사유": ", ".join(excluded) or "없음",
        "기대 결과": ", ".join(sorted(expected)) or "없음",
        "일치": accepted == expected
    })

context_validation = pd.DataFrame(validation_rows)
display(context_validation)

assert context_validation["일치"].all(), "문맥 검증 테스트 중 기대 결과와 다른 항목이 있습니다."
print("문맥 검증 테스트 전체 통과")


In [ ]:
# 6. 문장 분리
CONJ_WORDS = {"하지만", "그렇지만", "그치만", "허지만", "그러하지만"}


def split_sentences(text):
    base_sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?~])\s+|\n+", str(text))
        if sentence.strip()
    ]

    result = []
    for sentence in base_sentences:
        parts = ([part.strip() for part in re.split(r"(?<=지만)\s+", sentence) if part.strip()]
                 if SPLIT_CONNECTIVES else [sentence])

        carry = ""
        for part in parts:
            if part in CONJ_WORDS:
                carry = (carry + " " + part).strip()
                continue

            if carry:
                part = carry + " " + part
                carry = ""

            if part.endswith("지만"):
                part = part[:-2] + "다"
            result.append(part)

        if carry:
            result.append(carry)

    return result


for sample in [
    "오늘 시험을 잘 봐서 뿌듯했지만 다른 발표가 걱정되고 너무 피곤하다.",
    "시험을 망쳤다. 하지만 친구가 위로해줘서 괜찮았다. 그렇지만 내일이 걱정된다."
]:
    print(split_sentences(sample))


In [ ]:
# 7. 그룹별 강도 계산
def calculate_group_strengths(probabilities, top_k=GROUP_TOP_K):
    """각 문장에서 그룹 안 상위 k개 확률 평균을 구한 뒤 일기 전체에서 평균한다."""
    group_sentence_scores = {}
    group_strength = {}

    for group, indices in GROUP_IDX.items():
        values = probabilities[:, indices]
        k = min(top_k, values.shape[1])
        sorted_values = np.sort(values, axis=1)
        sentence_scores = sorted_values[:, -k:].mean(axis=1)
        group_sentence_scores[group] = sentence_scores
        group_strength[group] = float(sentence_scores.mean())

    return group_sentence_scores, group_strength


def polarity_score(strength, groups, aggregate=None):
    values = [strength[group] for group in groups]
    method = aggregate or POLARITY_AGG
    return float(np.max(values) if method == "max" else np.mean(values))


In [ ]:
# 8. 하루 일기 분석 — 대표 감정은 모델 기반, 명시적 감정은 다중 감정 보완
def analyze_text(text, aggregate=None):
    sentences = split_sentences(text)
    probabilities = predict_sentences(sentences)

    if len(probabilities) == 0:
        raise ValueError("분석할 문장이 없습니다.")

    model_picked = probabilities >= TH_VEC
    final_picked = model_picked.copy()

    mention_details_by_sentence = [context_mention_details(sentence) for sentence in sentences]
    accepted_by_sentence = [
        {item["idx"] for item in details if item["accepted"]}
        for details in mention_details_by_sentence
    ]

    if INCLUDE_VALID_MENTION_BELOW_THRESHOLD:
        for row_index, accepted in enumerate(accepted_by_sentence):
            if accepted:
                final_picked[row_index, list(accepted)] = True

    mean_probs = probabilities.mean(axis=0)
    group_sentence_scores, group_strength = calculate_group_strengths(probabilities)

    positive_score = polarity_score(group_strength, POSITIVE_GROUPS, aggregate)
    negative_score = polarity_score(group_strength, NEGATIVE_GROUPS, aggregate)
    neutral_score = float(group_strength["없음"])
    valence = positive_score - negative_score

    non_neutral_max = max(
        max(group_strength[group] for group in POSITIVE_GROUPS),
        max(group_strength[group] for group in NEGATIVE_GROUPS)
    )

    # 08: 혼합 판정 없이 대표 감정 하나를 유지
    if non_neutral_max < NEUTRAL_MIN_SCORE:
        main_group = "없음"
    elif positive_score >= negative_score:
        main_group = max(POSITIVE_GROUPS, key=lambda group: group_strength[group])
    else:
        main_group = max(NEGATIVE_GROUPS, key=lambda group: group_strength[group])

    main_emotion = DISPLAY_EMOTION[main_group]
    positive_main_group = max(POSITIVE_GROUPS, key=lambda group: group_strength[group])
    negative_main_group = max(NEGATIVE_GROUPS, key=lambda group: group_strength[group])
    positive_emotion = DISPLAY_EMOTION[positive_main_group]
    negative_emotion = DISPLAY_EMOTION[negative_main_group]

    # 문맥 검증을 통과한 직접 표현을 먼저, 그 뒤 모델 선택 감정을 확률 순으로 표시
    day_context_mentions = set().union(*accepted_by_sentence) if accepted_by_sentence else set()
    day_picked = final_picked.any(axis=0)
    ranked = list(rank_order(mean_probs))
    top_indices = (
        [idx for idx in ranked if idx in day_context_mentions]
        + [idx for idx in ranked if day_picked[idx] and idx not in day_context_mentions]
    )
    if not top_indices:
        top_indices = ranked
    top_emotions = [LABELS[idx] for idx in top_indices[:5]]

    sentence_rows = []
    for row_index, sentence in enumerate(sentences):
        accepted = accepted_by_sentence[row_index]
        details = mention_details_by_sentence[row_index]
        order = display_order(probabilities[row_index], sentence)

        selected = []
        for idx in order:
            if not final_picked[row_index, idx]:
                continue
            marker = "*" if idx in accepted else ""
            selected.append(f"{marker}{LABELS[idx]} {probabilities[row_index, idx]:.2f}")
            if len(selected) == 5:
                break

        sentence_strength = {
            group: float(group_sentence_scores[group][row_index])
            for group in GROUP_NAMES
        }
        sentence_positive = polarity_score(sentence_strength, POSITIVE_GROUPS, aggregate)
        sentence_negative = polarity_score(sentence_strength, NEGATIVE_GROUPS, aggregate)
        sentence_non_neutral = max(
            max(sentence_strength[group] for group in POSITIVE_GROUPS),
            max(sentence_strength[group] for group in NEGATIVE_GROUPS)
        )

        if sentence_non_neutral < NEUTRAL_MIN_SCORE:
            sentence_group = "없음"
        elif sentence_positive >= sentence_negative:
            sentence_group = max(POSITIVE_GROUPS, key=lambda group: sentence_strength[group])
        else:
            sentence_group = max(NEGATIVE_GROUPS, key=lambda group: sentence_strength[group])

        accepted_names = [LABELS[idx] for idx in sorted(accepted, key=lambda idx: -probabilities[row_index, idx])]
        excluded_names = [
            f"{item['emotion']}({item['reason']})"
            for item in details if not item["accepted"]
        ]

        sentence_rows.append({
            "sentence": sentence,
            "main_emotion": DISPLAY_EMOTION[sentence_group],
            "positive_score": round(sentence_positive, 3),
            "negative_score": round(sentence_negative, 3),
            "neutral_score": round(sentence_strength["없음"], 3),
            "direct_emotions": ", ".join(accepted_names) or "없음",
            "excluded_mentions": ", ".join(excluded_names) or "없음",
            "emotions": ", ".join(selected) if selected else "없음"
        })

    excluded_day = [
        f"{item['emotion']}({item['reason']})"
        for details in mention_details_by_sentence
        for item in details if not item["accepted"]
    ]

    day = {
        "n_sent": len(sentences),
        "positive_score": round(positive_score, 3),
        "negative_score": round(negative_score, 3),
        "neutral_score": round(neutral_score, 3),
        "valence": round(float(valence), 3),
        "positive_emotion": positive_emotion,
        "negative_emotion": negative_emotion,
        "main_group": main_group,
        "main_emotion": main_emotion,
        "direct_emotions": ", ".join(LABELS[idx] for idx in top_indices if idx in day_context_mentions) or "없음",
        "excluded_mentions": ", ".join(excluded_day) or "없음",
        "top_emotions": ", ".join(top_emotions)
    }

    for group in GROUP_NAMES:
        day[f"s_{group}"] = round(group_strength[group], 3)

    return day, sentence_rows, mean_probs


In [ ]:
# 9. 문맥 기반 분석 결과 테스트
CONTEXT_ANALYSIS_EXAMPLES = [
    "나는 지금 정말 행복하다.",
    "합격하면 행복할 것 같다.",
    "행복했으면 좋겠다.",
    "시험을 잘 봐서 정말 뿌듯했다.",
    "뿌듯할 줄 알았다.",
    "모든 게 절망적이었다.",
    "절망할 것 같았다.",
    "전혀 행복하지 않았다.",
    "친구가 행복해 보였다.",
    "예전에는 행복했지만 지금은 힘들다."
]

for text in CONTEXT_ANALYSIS_EXAMPLES:
    day, rows, _ = analyze_text(text)
    print("\n문장:", text)
    print("대표 감정:", day["main_emotion"])
    print("문맥 인정 직접 감정:", day["direct_emotions"])
    print("문맥 제외:", day["excluded_mentions"])
    print("세부 Top5:", day["top_emotions"])
    for row in rows:
        print(" -", row["sentence"], "→", row["emotions"])


In [ ]:
# 10. 일기 데이터 불러오기
if not DIARY_FILE.exists():
    raise FileNotFoundError(f"일기 파일이 없습니다: {DIARY_FILE}")

diary = pd.read_csv(DIARY_FILE, encoding="utf-8-sig")
diary["date"] = pd.to_datetime(diary["date"]).dt.date
diary = (
    diary.sort_values("date")
    .drop_duplicates("date", keep="last")
    .reset_index(drop=True)
)

print("일기 수:", len(diary))
print(diary["date"].min(), "~", diary["date"].max())
diary.head()


In [ ]:
# 11. 하루 코멘트 — 혼합 분기 제거
def daily_comment(row):
    emotion = row["main_emotion"]
    positive = row["positive_score"]
    negative = row["negative_score"]

    messages = {
        "행복": "긍정적인 감정이 더 강하게 나타난 하루예요. 좋았던 순간을 기억해두세요.",
        "기대": "앞으로의 일에 대한 기대와 관심이 비교적 강하게 나타난 하루예요.",
        "피로": "피로와 지침이 많이 나타난 하루예요. 충분한 휴식을 챙겨보세요.",
        "불안": "걱정과 불안 표현이 많이 나타난 하루예요.",
        "분노": "속상하거나 화가 나는 감정이 강하게 나타난 하루예요.",
        "슬픔": "슬픔이나 실망과 관련된 표현이 많이 나타난 하루예요.",
        "중립": "감정 변화가 비교적 잔잔한 하루였어요."
    }

    message = messages.get(emotion, "여러 감정이 나타난 하루였어요.")
    if row.get("direct_emotions", "없음") != "없음":
        message += f" 직접 표현된 감정: {row['direct_emotions']}."
    elif positive >= negative and emotion not in {"행복", "기대", "중립"}:
        message += " 긍정적인 감정도 함께 확인됐어요."
    return message


In [ ]:
# 12. 전체 일기 분석 + CSV 저장
def build_daily_outputs():
    day_rows = []
    sentence_rows = []

    for _, row in diary.iterrows():
        day_result, sentence_result, _ = analyze_text(row["text"])
        day_rows.append({"date": row["date"], "text": row["text"], **day_result})
        for sentence in sentence_result:
            sentence_rows.append({"date": row["date"], **sentence})

    daily_frame = pd.DataFrame(day_rows)
    daily_frame["comment"] = daily_frame.apply(daily_comment, axis=1)
    sentence_frame = pd.DataFrame(sentence_rows)

    daily_frame.to_csv(
        output_dir / "diary_daily_context_v3.csv",
        index=False,
        encoding="utf-8-sig"
    )
    sentence_frame.to_csv(
        output_dir / "diary_sentences_context_v3.csv",
        index=False,
        encoding="utf-8-sig"
    )
    return daily_frame, sentence_frame


daily, sentences = build_daily_outputs()
daily[[
    "date", "positive_score", "positive_emotion",
    "negative_score", "negative_emotion", "neutral_score",
    "main_emotion", "direct_emotions", "excluded_mentions",
    "valence", "top_emotions"
]].head()


In [ ]:
# 12-1. 긍정/부정 점수 방식 비교: mean vs max
comparison = []
for _, row in diary.iterrows():
    for aggregate in ["mean", "max"]:
        result, _, _ = analyze_text(row["text"], aggregate=aggregate)
        comparison.append({
            "date": row["date"],
            "agg": aggregate,
            "P": result["positive_score"],
            "N": result["negative_score"],
            "대표": result["main_emotion"]
        })

aggregate_compare = pd.DataFrame(comparison)
print("[대표 감정 일수]")
print(
    aggregate_compare.pivot_table(
        index="대표", columns="agg", values="date", aggfunc="count", fill_value=0
    ).reindex(DISPLAY_EMOTIONS).fillna(0).astype(int)
)
print("\n[평균 점수]")
print(aggregate_compare.groupby("agg")[["P", "N"]].mean().round(3))


In [ ]:
# 13. 특정 연/월 데이터 선택
def get_month_data(year, month):
    selected = daily[[
        day.year == year and day.month == month
        for day in daily["date"]
    ]].copy()
    return selected.sort_values("date").reset_index(drop=True)

last_date = daily["date"].max()
SELECT_YEAR = last_date.year
SELECT_MONTH = last_date.month
print("기본 표시 월:", SELECT_YEAR, SELECT_MONTH)
print("해당 월 일기 수:", len(get_month_data(SELECT_YEAR, SELECT_MONTH)))


In [ ]:
# 14. 캘린더 공통 그리기 함수
def draw_month_calendar(year, month, value_column, emotion_column, title, filename, mode="emotion"):
    month_data = get_month_data(year, month)
    by_date = month_data.set_index("date")
    weeks = calendar.Calendar(firstweekday=0).monthdatescalendar(year, month)

    fig, ax = plt.subplots(figsize=(10, 1.5 + 1.25 * len(weeks)))

    for column, name in enumerate("월화수목금토일"):
        ax.text(
            column + 0.5, -0.3, name,
            ha="center", va="center", fontsize=11,
            color="#D33" if column == 6 else "#36C" if column == 5 else "#333"
        )

    for row, week in enumerate(weeks):
        for column, day in enumerate(week):
            inside = day.month == month
            has_diary = inside and day in by_date.index

            if not inside:
                facecolor, alpha = "white", 1.0
            elif not has_diary:
                facecolor, alpha = "#F5F5F5", 1.0
            else:
                emotion = by_date.loc[day, emotion_column]
                score = float(by_date.loc[day, value_column])
                facecolor = EMOTION_COLORS[emotion]
                alpha = 0.85 if mode == "emotion" else 0.20 + 0.70 * np.clip(score, 0, 1)

            ax.add_patch(Rectangle(
                (column + 0.04, row + 0.04), 0.92, 0.92,
                facecolor=facecolor, alpha=alpha, edgecolor="#DDD"
            ))

            if inside:
                ax.text(
                    column + 0.1, row + 0.2, str(day.day), fontsize=10,
                    color="#222" if has_diary else "#AAA",
                    fontweight="bold" if has_diary else "normal"
                )

            if has_diary:
                emotion = by_date.loc[day, emotion_column]
                score = float(by_date.loc[day, value_column])
                ax.text(
                    column + 0.5, row + 0.55, emotion,
                    ha="center", va="center", fontsize=9, fontweight="bold"
                )
                if mode != "emotion":
                    ax.text(column + 0.5, row + 0.80, f"{score:.2f}",
                            ha="center", va="center", fontsize=7)

    ax.set_xlim(0, 7)
    ax.set_ylim(len(weeks), -0.6)
    ax.axis("off")
    ax.set_title(f"{year}년 {month}월 {title}", fontsize=15, pad=12)
    plt.tight_layout()
    fig.savefig(output_dir / filename, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# 15. 한 달 감정 점수 그래프
def plot_month_score_graph(year, month, score_column, title, color, filename):
    month_data = get_month_data(year, month)
    fig, ax = plt.subplots(figsize=(11, 4))

    if not month_data.empty:
        days = np.array([day.day for day in month_data["date"]], dtype=int)
        scores = month_data[score_column].astype(float).to_numpy()
        ax.plot(days, scores, marker="o", markersize=5, linewidth=2, color=color)
        ax.scatter(days, scores, color=color, s=30, zorder=3)
    else:
        ax.text(0.5, 0.5, "해당 월에 작성된 일기가 없습니다.",
                ha="center", va="center", transform=ax.transAxes)

    ax.set_xlim(1, calendar.monthrange(year, month)[1])
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("날짜 (일)")
    ax.set_ylabel("감정 강도 (0~1)")
    ax.set_title(f"{year}년 {month}월 {title}")
    ax.grid(True, linestyle="--", alpha=0.35)
    fig.tight_layout()
    fig.savefig(output_dir / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_positive_graph(year, month):
    plot_month_score_graph(
        year, month, "positive_score", "긍정 감정 추이", "#2E9D68",
        f"graph_positive_context_{year}_{month:02d}.png"
    )


def plot_negative_graph(year, month):
    plot_month_score_graph(
        year, month, "negative_score", "부정 감정 추이", "#D85A56",
        f"graph_negative_context_{year}_{month:02d}.png"
    )


plot_positive_graph(SELECT_YEAR, SELECT_MONTH)
plot_negative_graph(SELECT_YEAR, SELECT_MONTH)


In [ ]:
# 16. 한 달 전체 감정 캘린더 — 혼합 색상 없음
def plot_emotion_calendar(year, month):
    draw_month_calendar(
        year=year,
        month=month,
        value_column="valence",
        emotion_column="main_emotion",
        title="감정 캘린더",
        filename=f"calendar_emotion_context_{year}_{month:02d}.png",
        mode="emotion"
    )

    handles = [Patch(color=EMOTION_COLORS[emotion], label=emotion)
               for emotion in DISPLAY_EMOTIONS]
    fig, ax = plt.subplots(figsize=(8, 0.8))
    ax.axis("off")
    ax.legend(handles=handles, loc="center", ncol=7, frameon=False)
    plt.show()


plot_emotion_calendar(SELECT_YEAR, SELECT_MONTH)


In [ ]:
# 17. 지정한 달의 그래프 2개 + 전체 감정 캘린더
def show_month_summary(year, month):
    print(f"{year}년 {month}월")
    print("1) 긍정 감정 그래프")
    plot_positive_graph(year, month)
    print("2) 부정 감정 그래프")
    plot_negative_graph(year, month)
    print("3) 전체 감정 캘린더")
    plot_emotion_calendar(year, month)


In [ ]:
# 18. 오늘 일기 쓰기 + 분석 결과 자동 갱신
def write_diary(text, day=None, save=True, refresh=True):
    global diary, daily, sentences

    day = day or date.today()
    result, sentence_rows, _ = analyze_text(text)

    print(f"[{day}]")
    print("(* = 문맥 검증을 통과한 직접 감정 표현)")

    for row in sentence_rows:
        print(f"- {row['sentence']}")
        print(
            f"  → 대표: {row['main_emotion']} | "
            f"P {row['positive_score']:.3f} / N {row['negative_score']:.3f} / "
            f"중립 {row['neutral_score']:.3f}"
        )
        print(f"  → 세부: {row['emotions']}")
        if row["direct_emotions"] != "없음":
            print(f"  → 직접 감정 인정: {row['direct_emotions']}")
        if row["excluded_mentions"] != "없음":
            print(f"  → 문맥상 제외: {row['excluded_mentions']}")

    print("\nPositive:", result["positive_score"], "|", result["positive_emotion"])
    print("Negative:", result["negative_score"], "|", result["negative_emotion"])
    print("Neutral (없음):", result["neutral_score"])
    print("대표 감정:", result["main_emotion"])
    print("Valence:", result["valence"])
    print("직접 감정 인정:", result["direct_emotions"])
    print("문맥상 제외:", result["excluded_mentions"])
    print("세부 Top5:", result["top_emotions"])

    temporary = {**result}
    temporary["comment"] = daily_comment(temporary)
    print("코멘트:", temporary["comment"])

    if not save:
        return temporary

    old = pd.read_csv(DIARY_FILE, encoding="utf-8-sig")
    old = old[old["date"] != str(day)]
    new_row = pd.DataFrame([{"date": str(day), "text": text}])
    old = pd.concat([old, new_row], ignore_index=True).sort_values("date")
    old.to_csv(DIARY_FILE, index=False, encoding="utf-8-sig")
    print("\n일기 저장 완료")

    if refresh:
        diary = pd.read_csv(DIARY_FILE, encoding="utf-8-sig")
        diary["date"] = pd.to_datetime(diary["date"]).dt.date
        diary = diary.sort_values("date").reset_index(drop=True)
        daily, sentences = build_daily_outputs()
        print("분석 결과 자동 갱신 완료")
        show_month_summary(day.year, day.month)

    return temporary


In [ ]:
# 19. 최종 테스트 — 대표 감정 하나 + 문맥 검증형 직접 감정
result = write_diary(
    "오늘 시험을 잘 봐서 뿌듯했지만 다른 발표가 걱정되고 너무 피곤하다.",
    save=False
)


In [ ]:
# 20. 긴 일기 테스트 — 파일이 있을 때만 실행
long_diary_path = data_dir / "test_diary_long.txt"
if long_diary_path.exists():
    long_text = long_diary_path.read_text(encoding="utf-8")
    result = write_diary(long_text, save=False)
else:
    print("긴 일기 테스트 생략: 파일이 없습니다 →", long_diary_path)
